In [2]:
import os
# --- GPU memory: MUST be set BEFORE importing jax ---
# Default JAX grabs ~75% of VRAM on first op and never returns it. Across many Optuna
# trials (each builds a fresh jit(vmap(...)) over a new env/agent) the compiled programs
# + device buffers pile up until one allocation OOMs the kernel -> the crash you saw on
# trial 1+. preallocate=false makes JAX allocate on demand AND release between trials
# (paired with the jax.clear_caches()+gc.collect() added to each objective's finally).
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
# Optional hard cap on this kernel's slice (uncomment if you still see OOM with 3 seeds):
# os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.6")
import sys
import gc
import jax
import jax.numpy as jnp
import wandb
import optuna
import time
from pathlib import Path
from dataclasses import fields

# Loco Mujoco
from loco_mujoco import TaskFactory
from loco_mujoco.algorithms import PPOJax
from loco_mujoco.utils.metrics import QuantityContainer
from loco_mujoco.utils import MetricsHandler
from loco_mujoco.trajectory.dataclasses import Trajectory
from loco_mujoco.task_factories.dataset_confs import CustomDatasetConf

# Hydra Compose API (Especial para Jupyter)
from hydra import initialize_config_dir, compose
from simulation.config import paths
from omegaconf import OmegaConf

## Algorythm Parameters

In [3]:
# CELDA 2: CARGAR CONF.YAML
# ==========================================
# ==========================================
# Inicializamos Hydra apuntando a la carpeta actual ("./")
try:
    initialize_config_dir(version_base=None, config_dir=str(paths.CONF_DIR))
except ValueError:
    # Captura el error si ejecutas esta celda dos veces (Hydra ya estaba inicializado)
    pass

# Cargamos tu conf.yaml
base_config = compose(config_name="conf")

print("Configuración base cargada exitosamente.")
# Opcional: print(OmegaConf.to_yaml(base_config)) para verla

Configuración base cargada exitosamente.


In [ ]:
# ==========================================
# FASE 1: FUNCIÓN OBJETIVO (HIPERPARÁMETROS PPO)
# ==========================================
# Proyecto de WandB SOLO para la Fase 1 (separado del entrenamiento y de la Fase 2).
PHASE1_WANDB_PROJECT = "deep_mimimic_senecabot1"

# Baseline = agente 18-41-27 (euclidean_distance final de validación). El objetivo está
# normalizado a estos valores -> cada término ~1.0 en el baseline; total baseline = 6.0,
# así que un valor < 6.0 es MEJOR que el agente actual.
BASE = dict(qpos=2.48, qvel=2.53, site_rpos=0.083, site_rrotvec=0.129, site_rvel=4.88)
# qpos pesa 2x: es el objetivo de imitación y la métrica limitante (se estanca ~14M y no baja
# con más entrenamiento). El resto pesa 1x.
W    = dict(qpos=2.0,  qvel=1.0,  site_rpos=1.0,   site_rrotvec=1.0,   site_rvel=1.0)

# --- SUAVIDAD (DTW): el Dynamic Time Warping mide desajuste de FORMA/FASE del ciclo de marcha,
# que la distancia euclidiana por-frame NO ve (una política que tiembla alrededor de la
# referencia puntúa bien en euclidiana pero es jittery). DTW ya se computa en cada validación
# (conf.validation.measures lo incluye); aquí solo lo aprovechamos -> coste cero.
# OJO: el agente baseline 18-41-27 no está local, así que NO tenemos su DTW. Por eso DTW
# arranca SOLO logueado (W_DTW=0, no entra al objetivo). Para activarlo: tras ~8 trials random
# leer la mediana de 'sweep/dtw_qpos' / 'sweep/dtw_qvel', ponerla en BASE_DTW y subir W_DTW.
BASE_DTW = dict(qpos=1.0, qvel=1.0)          # PLACEHOLDER -> medir/calibrar antes de activar
W_DTW    = dict(qpos=0.0, qvel=0.0)          # 0 = logueado pero fuera del objetivo (sin sesgo)

# --- CONVERGENCIA / ESTABILIDAD: ~0 para un agente que converge y se queda estable (no mueve
# el baseline de 6.0); castiga configs que NO convergen / colapsan / se mueven a tirones.
LAM_SURV = 1.0   # supervivencia : max(0, 1 - eplen/horizon)         -> muere antes = inestable
LAM_REG  = 0.5   # regresión     : max(0, qpos_final - qpos_best)/BASE-> colapsa tras converger
LAM_VAR  = 0.5   # varianza cola : std(qpos[-K:])/BASE               -> sigue rebotando, no convergió

# --- PROXY + PRUNING: 15M en N_CHUNKS tramos (resume) para poder PODAR entre tramos. El
# entrenamiento es un único jax.lax.scan no interrumpible, así que la única forma de podar
# (sin tocar loco-mujoco-seneca) es entrenar en tramos y decidir en Python entre uno y otro.
PROXY_STEPS = 15_000_000
N_CHUNKS    = 3
VAL_PER_CHUNK = 4    # validaciones por tramo (deja cola K=3 para el composite del último tramo)


def objective(trial):
    import numpy as np
    # 1. Copia editable de la config base
    config = base_config.copy()
    OmegaConf.set_struct(config, False)

    # --- ROBUSTEZ A LA SEMILLA (evita sobreajustar la HPO a una sola instancia de azar) ---
    config.experiment.n_seeds = 1          # 3 semillas en paralelo (vmap); se promedian abajo
    config.experiment.num_envs = 2048      # salvaguarda VRAM: 3 semillas ~3x memoria (sube si cabe)

    # --- A. ESPACIO DE BÚSQUEDA ---
    # Core PPO
    config.experiment.lr            = trial.suggest_float("lr", 5e-5, 8e-4, log=True)
    config.experiment.clip_eps      = trial.suggest_float("clip_eps", 0.1, 0.3)
    config.experiment.update_epochs = trial.suggest_int("update_epochs", 3, 10)
    config.experiment.gamma         = trial.suggest_categorical("gamma", [0.98, 0.99, 0.995])
    config.experiment.gae_lambda    = trial.suggest_float("gae_lambda", 0.9, 0.99)
    # Exploración / actuación: la política actual es PASIVA (acciones ~0, sin saturación),
    # así que abrimos las perillas que controlan cuánto explora/actúa.
    config.experiment.ent_coef      = trial.suggest_float("ent_coef", 1e-4, 3e-2, log=True)
    config.experiment.init_std      = trial.suggest_float("init_std", 0.2, 1.0)

    # --- B. PROXY EN TRAMOS: total por tramo = PROXY_STEPS / N_CHUNKS (num_updates se calcula
    # por-tramo dentro de init_agent_conf, así que cada llamada entrena un tramo). ---
    config.experiment.total_timesteps = PROXY_STEPS // N_CHUNKS
    config.experiment.validation.num  = VAL_PER_CHUNK

    # --- C. WANDB ---
    config_dict = OmegaConf.to_container(config, resolve=True, throw_on_missing=True)
    run = wandb.init(
        project=PHASE1_WANDB_PROJECT,
        config=config_dict,
        name=f"p1_trial_{trial.number}",
        group="optuna_phase1",
        settings=wandb.Settings(quiet=True),
    )
    # Curva de retorno de episodio en su PROPIO eje x ("train/update") para que las llamadas
    # a run.log puedan no llevar step= y el step global de wandb quede monótono (no pelea con
    # las métricas de pruning/sweep, que van en sus propios ejes).
    run.define_metric("train/update")
    run.define_metric("train/mean_episode_return", step_metric="train/update")
    run.define_metric("prune/chunk")
    run.define_metric("prune/*", step_metric="prune/chunk")

    result_dir = str(paths.OPTUNA_DIR / f"phase1_trial_{trial.number}")
    os.makedirs(result_dir, exist_ok=True)

    # Locales pesados inicializados para que el finally pueda liberarlos siempre (también si
    # falla antes de crearlos) -> evita la acumulación de memoria GPU entre trials.
    env = agent_conf = mh = train_fn = resume_fn = out = agent_state = None
    traj = custom_conf = factory = None

    try:
        # --- LÓGICA DE LOCO MUJOCO ---
        factory = TaskFactory.get_factory_cls(config.experiment.task_factory.name)

        file_path = str(paths.TRAJ_ADAPTED)
        traj = Trajectory.load(file_path)
        custom_conf = CustomDatasetConf(traj=traj)

        env = factory.make(**config.experiment.env_params, custom_dataset_conf=custom_conf)
        agent_conf = PPOJax.init_agent_conf(env, config)
        mh = MetricsHandler(config, env) if config.experiment.validation.active else None

        # train_fn = tramo 0 (desde cero); resume_fn = tramos siguientes (parte del agent_state
        # anterior). Mismas shapes en todos los tramos -> resume_fn compila una sola vez.
        train_fn  = PPOJax.build_train_fn(env, agent_conf, mh=mh)
        resume_fn = PPOJax.build_resume_train_fn(env, agent_conf, mh=mh)
        if config.experiment.n_seeds > 1:
            train_fn  = jax.jit(jax.vmap(train_fn))
            resume_fn = jax.jit(jax.vmap(resume_fn))
        else:
            train_fn  = jax.jit(train_fn)
            resume_fn = jax.jit(resume_fn)

        def _seed_keys(chunk):
            # (n_seeds, 2) para vmap, o (2,) para una sola semilla; clave distinta por tramo.
            keys = [jax.random.PRNGKey(1000 * chunk + i) for i in range(config.experiment.n_seeds + 1)]
            return jnp.squeeze(jnp.vstack(keys[1:]))

        # --- ENTRENAMIENTO POR TRAMOS + PRUNING ---
        g_update = 0   # índice global de update (continuo a través de los tramos) para la curva
        for c in range(N_CHUNKS):
            out = train_fn(_seed_keys(c)) if c == 0 else resume_fn(_seed_keys(c), agent_state)
            agent_state = out["agent_state"]

            # (n_seeds, num_updates_tramo); promedio sobre semillas -> curva por update.
            ret = np.atleast_2d(np.asarray(out["training_metrics"].mean_episode_return))
            ret_curve = np.nanmean(ret, axis=0)
            for v in ret_curve:
                run.log({"train/update": g_update, "train/mean_episode_return": float(v)})
                g_update += 1

            # Señal de pruning = retorno de episodio (= recompensa de imitación, porque
            # proportion_env_reward=0). El estudio MINIMIZA -> reportamos el NEGATIVO
            # (más retorno = mejor = valor menor). Es una señal barata y temprana, ya
            # disponible sin tocar la librería (entropy NO se devuelve, solo va a wandb).
            ret_tail = float(np.nanmean(ret[:, -5:]))     # media de las últimas updates, sobre semillas
            # Si no terminó ningún episodio en la cola (mean=0/0=NaN) NO reportamos este tramo:
            # reportar NaN rompe el pruner, y "sin dones todavía" no implica que el trial sea malo.
            if np.isfinite(ret_tail):
                trial.report(-ret_tail, step=c)
                run.log({"prune/chunk": c, "prune/neg_return": -ret_tail, "prune/ep_return": ret_tail})
                if trial.should_prune():
                    raise optuna.TrialPruned()

        # === MÉTRICAS DEL ÚLTIMO TRAMO -> COMPOSITE (robusto a la semilla) ===
        # out["validation_metrics"]: cada hoja (n_seeds, num_updates) con vmap, o (num_updates,).
        ed  = out["validation_metrics"].euclidean_distance
        dtw = out["validation_metrics"].dynamic_time_warping

        K = 3
        def series_per_seed(arr):
            # serie de validación por semilla, descartando los slots vacíos (==0).
            a = np.atleast_2d(np.asarray(arr))            # (n_seeds, num_updates)
            return [row[row != 0] for row in a]

        def tail_mean(series):                            # media de los últimos K (la val es ruidosa)
            return [float(np.mean(s[-K:])) if s.size else float("nan") for s in series]

        ed_series  = {k: series_per_seed(getattr(ed,  k)) for k in BASE}
        dtw_series = {k: series_per_seed(getattr(dtw, k)) for k in BASE_DTW}
        ed_tail    = {k: tail_mean(ed_series[k])  for k in BASE}
        dtw_tail   = {k: tail_mean(dtw_series[k]) for k in BASE_DTW}

        # --- términos de convergencia/estabilidad (sobre la serie de qpos, la métrica limitante) ---
        def regress(series):   # final peor que el mejor -> colapso post-convergencia
            return [max(0.0, float(s[-1] - s.min())) if s.size else float("nan") for s in series]
        def tailstd(series):   # cola que rebota -> no convergió
            return [float(np.std(s[-K:])) if s.size else float("nan") for s in series]
        qpos_regress = regress(ed_series["qpos"])
        qpos_tailstd = tailstd(ed_series["qpos"])

        horizon = float(config.experiment.env_params.horizon)
        eplen = np.atleast_2d(np.asarray(out["training_metrics"].mean_episode_length))
        eplen_tail   = [float(np.nanmean(r[-5:])) for r in eplen]
        survival_pen = [max(0.0, 1.0 - e / horizon) if e == e else float("nan") for e in eplen_tail]

        # --- composite POR SEMILLA (una semilla con NaN -> 9999: castiga inestabilidad) ---
        n_eff = len(ed_tail["qpos"])
        comps = []
        for s in range(n_eff):
            vals = ([ed_tail[k][s] for k in BASE] + [dtw_tail[k][s] for k in BASE_DTW]
                    + [qpos_regress[s], qpos_tailstd[s], survival_pen[s]])
            if any(v != v for v in vals):
                comps.append(9999.0); continue
            acc    = sum(W[k]     * ed_tail[k][s]  / BASE[k]     for k in BASE)
            smooth = sum(W_DTW[k] * dtw_tail[k][s] / BASE_DTW[k] for k in BASE_DTW)   # 0 hasta calibrar
            stab   = (LAM_SURV * survival_pen[s]
                      + LAM_REG * qpos_regress[s] / BASE["qpos"]
                      + LAM_VAR * qpos_tailstd[s] / BASE["qpos"])
            comps.append(acc + smooth + stab)
        comps = np.array(comps)

        # Objetivo = media + 0.5*std entre semillas (robustez: castiga alta-media-pero-inestable).
        composite = float(comps.mean() + 0.5 * comps.std())

        # --- sub-métricas medias para re-pesar post-hoc sin re-entrenar ---
        m = {k: float(np.nanmean(ed_tail[k]))  for k in BASE}
        m.update({f"dtw_{k}": float(np.nanmean(dtw_tail[k])) for k in BASE_DTW})
        m["qpos_regress"] = float(np.nanmean(qpos_regress))
        m["qpos_tailstd"] = float(np.nanmean(qpos_tailstd))
        m["survival_pen"] = float(np.nanmean(survival_pen))
        for k, v in m.items():
            trial.set_user_attr(f"sweep/{k}", v)
        trial.set_user_attr("sweep/composite_std", float(comps.std()))

        run.log({"Metric for Sweep": composite,
                 "sweep/composite_mean": float(comps.mean()),
                 "sweep/composite_std": float(comps.std()),
                 **{f"sweep/{k}": v for k, v in m.items()}})
        wandb.finish()

        return composite

    except optuna.TrialPruned:
        # Importante: re-lanzar para que Optuna marque el trial como PRUNED (no COMPLETE/9999),
        # si no, el except genérico de abajo lo capturaría y contaminaría la mediana y el TPE.
        wandb.finish(exit_code=1)
        raise

    except Exception as e:
        wandb.finish(exit_code=1)
        print(f"Trial {trial.number} falló: {e}")
        return 9999.0

    finally:
        # Libera la memoria GPU de este trial para que NO se acumule entre trials (la causa
        # del crash del kernel). del de las referencias + clear_caches (suelta los ejecutables
        # compilados de este jit(vmap)) + gc para forzar la liberación de los buffers de device.
        out = agent_state = train_fn = resume_fn = env = agent_conf = mh = None
        traj = custom_conf = factory = None
        jax.clear_caches()
        gc.collect()

In [ ]:
# ==========================================
# FASE 1: EJECUTAR OPTIMIZACIÓN
# ==========================================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

print("Iniciando la búsqueda de hiperparámetros (Fase 1: PPO)...")

# TPE con semilla (reproducible) + arranque aleatorio para sembrar bien el espacio (7 dims).
sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=8)

# PRUNER: corta trials que ya pintan peor que la mediana a la misma altura de entrenamiento,
# usando el retorno de episodio reportado al final de cada uno de los N_CHUNKS tramos.
#   - n_startup_trials=8 : no poda hasta tener 8 trials COMPLETOS (mediana fiable; igual que el TPE).
#   - n_warmup_steps=1   : nunca poda en el tramo 0 (el más ruidoso, ~5M); poda desde el tramo 1.
# Un trial que diverge / colapsa (NaN -> retorno bajo) se corta temprano -> es además un
# filtro automático de NO-convergencia, alineado con el objetivo suave/convergente.
pruner = optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1)

# Persistencia SQLite: los trials sobreviven a caídas/reinicios (load_if_exists=True permite
# reanudar el mismo estudio). Métrica = error normalizado -> MINIMIZAR.
study = optuna.create_study(
    direction="minimize",
    study_name="ppo_phase1",
    storage=f"sqlite:///{paths.OPTUNA_DIR / 'optuna_phase1.db'}",
    load_if_exists=True,
    sampler=sampler,
    pruner=pruner,
)

# ~7 hiperparámetros -> 35 trials (15 es insuficiente para 7 dims con TPE). Con pruning el
# coste efectivo baja: los trials malos se cortan tras 1-2 tramos en vez de los 3 completos.
study.optimize(objective, n_trials=35)

# Resumen incluyendo cuántos se podaron (sanity check del pruner).
pruned   = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
print("\n¡Búsqueda Finalizada!")
print(f"Trials completos: {len(complete)} | podados: {len(pruned)}")
print(f"Mejor métrica compuesta: {study.best_value:.4f}   (baseline = 6.0; menor es mejor)")
print("Mejores hiperparámetros encontrados:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")

In [5]:
# ==========================================
# CELDA 5: GRÁFICOS INTERACTIVOS
# ==========================================
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate

# 1. Ver cómo fue bajando el error intento tras intento
fig1 = plot_optimization_history(study)
fig1.show()

# 2. Ver qué hiperparámetro tuvo el mayor impacto en el rendimiento
fig2 = plot_param_importances(study)
fig2.show()

# 3. Ver las combinaciones de hiperparámetros
fig3 = plot_parallel_coordinate(study)
fig3.show()

In [6]:
best_params = study.best_params
print("\nMejores hiperparámetros encontrados:")
for key, value in best_params.items():
    print(f"  - {key}: {value}") 


Mejores hiperparámetros encontrados:
  - lr: 0.00010474076398937535
  - clip_eps: 0.1872449833323218
  - update_epochs: 7
  - gamma: 0.98


## REWARD FUNCTION PARAMETERS

In [4]:
# ==========================================
# CELDA 3: FUNCIÓN OBJETIVO (FASE 2 - RECOMPENSAS)  [v3: debug_agent + altura/paso]
# ==========================================
# Proyecto de WandB SOLO para la Fase 2 (separado del entrenamiento y de la Fase 1).
PHASE2_WANDB_PROJECT = "deep_mimimic_senecabot2"

# === POR QUÉ CAMBIÓ LA "Metric for Sweep" ===
# La métrica anterior era la EuclideanDistance de validación (qpos, qvel, site_*), que es UN
# escalar promediado sobre TODAS las articulaciones (metrics.py: mean=True + jnp.mean). El
# diagnóstico de debug_agent sobre el mejor agente actual (2026-06-09/21-02-06) muestra que el
# problema NO es el promedio (9.7° global) sino unas pocas articulaciones rezagadas:
#     fl_ankle 19.6°, br_knee 14.2°, fl_knee 10.8°   vs   bl_ankle 5.0°
# El promedio euclidiano de validación es CIEGO a eso. Por eso la métrica ahora se construye con
# `agent_diagnostics.evaluate_agent` (reutiliza el rollout verificado de debug_agent) y pesa
# explícitamente las PEORES articulaciones, además de poner guardarraíles de estabilidad
# (supervivencia + muertes por inestabilidad/caída) que la métrica vieja ignoraba.
#
# Composite v3 (por semilla, MINIMIZAR; ver pesos/baseline en agent_diagnostics.py):
#     1.0·(joint_rms_mean/B) + 2.0·(joint_rms_worst/B) + 0.5·(act_rate/B)
#   + 1.0·max(0, 1−surv)     + 2.0·bad_death_frac
#   + 1.5·(height_rms/B)     + 1.0·|1 − speed_ratio|/B      [v3: castiga andar agachado y a medio paso]
# Baseline = el agente 13-24-37 medido con este mismo módulo -> composite ≈ 6.03.
# Un valor < 6.03 es MEJOR que ese agente; un agente que no avanza NO puede ganar este estudio
# (la lección del run 2026-06-11/13-40-31: trial #28 de v2 produjo speed_ratio 0.22).
from simulation.analysis.agent_diagnostics import evaluate_agent, composite_from_scalars, BASELINE

# Rollout de diagnóstico por trial (igual que debug_agent): determinista, 64 envs x 300 pasos.
DIAG_N_ENVS = 64
DIAG_N_STEPS = 300

# Proxy de 20M pasos por trial (qpos se estanca ~14M, así que 20M ya rankea el tracking
# articular con margen). NOTA: antes estaba en 100_000_000 -> buffers de device enormes
# (train_state_buffer + métricas) que contribuían al OOM/crash del kernel entre trials.
PROXY_STEPS = 20_000_000


def objective(trial):
    import numpy as np
    # 1. Copia editable de la config base (ya contiene los PPO params ganadores de la Fase 1)
    config = base_config.copy()
    OmegaConf.set_struct(config, False)

    # --- ROBUSTEZ A LA SEMILLA (evita sobreajustar la HPO a una sola instancia de azar) ---
    config.experiment.n_seeds = 1          # 3 semillas en paralelo (vmap); se promedian abajo
    config.experiment.num_envs = 2048      # salvaguarda VRAM: 3 semillas ~3x memoria (sube si cabe)

    # --- A. ESPACIO DE BÚSQUEDA: 9 PARÁMETROS DE RECOMPENSA (DeepMimic) ---
    rp = config.experiment.env_params.reward_params
    # 1. Pesos principales (posiciones) + reshape del gradiente de error articular (qpos_exp).
    #    qpos_w_sum / qpos_w_exp son las perillas que más afectan el tracking articular, que es
    #    justo lo que la nueva métrica prioriza (peores articulaciones).
    rp.qpos_w_sum         = trial.suggest_float("qpos_w",  0.1, 0.8)
    rp.rpos_w_sum         = trial.suggest_float("rpos_w",  0.1, 0.8)
    rp.qpos_w_exp         = trial.suggest_float("qpos_exp", 2.0, 10.0)
    # 2. Pesos secundarios (orientación y velocidades); techos reducidos para que no dominen
    rp.rquat_w_sum        = trial.suggest_float("rquat_w", 0.05, 0.3)
    rp.qvel_w_sum         = trial.suggest_float("qvel_w",  0.05, 0.4)
    rp.rvel_w_sum         = trial.suggest_float("rvel_w",  0.02, 0.15)
    # 2b. [v3] Exponente del término qvel: el default 2.0 satura exp(-e·dist) -> gradiente
    #     muerto (diagnóstico 06-09/06-10). Rango log para que el TPE pueda revivir el término.
    rp.qvel_w_exp         = trial.suggest_float("qvel_exp", 0.01, 2.0, log=True)
    # 3. Penalizadores de energía/suavidad (act_rate impacta directo en act_rate de la métrica)
    rp.action_rate_coeff  = trial.suggest_float("act_rate", 1e-4, 5e-2, log=True)
    rp.joint_torque_coeff = trial.suggest_float("torque",   1e-5, 1e-2, log=True)

    # --- B. PROXY TASK ---
    config.experiment.total_timesteps = PROXY_STEPS
    # NOTA: la validación interna se deja ACTIVA (como en el estudio original). La métrica ya no
    # lee sus salidas (se mide con el rollout de diagnóstico post-entrenamiento), pero `ppo_jax`
    # NO honra `validation.active` y calcula `validation_interval = num_updates // validation.num`
    # incondicionalmente, así que apagarla no quita overhead y sí arriesga romper el train_fn
    # (mh=None). Requisito implícito: num_updates >= validation.num (con 20M y 1024 envs -> ~97).

    # --- C. WANDB ---
    config_dict = OmegaConf.to_container(config, resolve=True, throw_on_missing=True)
    run = wandb.init(
        project=PHASE2_WANDB_PROJECT,
        config=config_dict,
        name=f"p2_trial_{trial.number}",
        group="optuna_phase2",
        settings=wandb.Settings(quiet=True),
    )
    # Curva de retorno de episodio en su propio eje x (igual que la Fase 1).
    run.define_metric("train/update")
    run.define_metric("train/mean_episode_return", step_metric="train/update")

    result_dir = str(paths.OPTUNA_DIR / f"phase2_trial_{trial.number}")
    os.makedirs(result_dir, exist_ok=True)

    # Locales pesados inicializados para que el finally pueda liberarlos siempre (también si
    # falla antes de crearlos) -> evita la acumulación de memoria GPU entre trials.
    env = agent_conf = mh = train_fn = out = agent_state = None
    traj = custom_conf = factory = None

    try:
        # --- LÓGICA DE LOCO MUJOCO (idéntica a la Fase 1) ---
        factory = TaskFactory.get_factory_cls(config.experiment.task_factory.name)

        file_path = str(paths.TRAJ_ADAPTED)
        traj = Trajectory.load(file_path)
        custom_conf = CustomDatasetConf(traj=traj)

        env = factory.make(**config.experiment.env_params, custom_dataset_conf=custom_conf)
        agent_conf = PPOJax.init_agent_conf(env, config)
        mh = MetricsHandler(config, env) if config.experiment.validation.active else None

        train_fn = PPOJax.build_train_fn(env, agent_conf, mh=mh)
        train_fn = jax.jit(jax.vmap(train_fn)) if config.experiment.n_seeds > 1 else jax.jit(train_fn)

        rngs = [jax.random.PRNGKey(i) for i in range(config.experiment.n_seeds + 1)]
        rng, _rng = rngs[0], jnp.squeeze(jnp.vstack(rngs[1:]))

        # EJECUCIÓN
        out = train_fn(_rng)
        agent_state = out["agent_state"]   # (con vmap: cada hoja lleva el eje de semilla delante)

        # --- Curva de retorno de episodio a wandb (promedio sobre semillas, por update) ---
        ret = np.atleast_2d(np.asarray(out["training_metrics"].mean_episode_return))
        ret_curve = np.nanmean(ret, axis=0)
        for i, v in enumerate(ret_curve):
            run.log({"train/update": i, "train/mean_episode_return": float(v)})

        # === MÉTRICA = DIAGNÓSTICO ESTILO debug_agent (por semilla) ===
        # evaluate_agent hace un rollout determinista por semilla y devuelve escalares:
        #   surv, eplen_frac, joint_rms_mean, joint_rms_worst, joint_rms_max,
        #   bad_death_frac, act_rate, height_rms, speed_ratio, per_joint_rms(12).
        per_seed = evaluate_agent(env, agent_conf, agent_state,
                                  n_envs=DIAG_N_ENVS, n_steps=DIAG_N_STEPS, deterministic=True)

        # Composite por semilla (NaN -> 9999 dentro de composite_from_scalars: castiga inestable).
        comps = np.array([composite_from_scalars(m) for m in per_seed])

        # Objetivo = media + 0.5*std entre semillas (robustez: castiga alta-media-pero-inestable).
        # Con 1 semilla std=0 -> == media.
        composite = float(comps.mean() + 0.5 * comps.std())

        # --- sub-métricas medias (sobre semillas) para inspección / re-pesar post-hoc ---
        keys = ["surv", "eplen_frac", "joint_rms_mean", "joint_rms_worst", "joint_rms_max",
                "bad_death_frac", "act_rate", "height_rms", "speed_ratio"]
        m = {k: float(np.nanmean([s[k] for s in per_seed])) for k in keys}
        for k, v in m.items():
            trial.set_user_attr(f"sweep/{k}", v)
        trial.set_user_attr("sweep/composite_std", float(comps.std()))

        run.log({"Metric for Sweep": composite,
                 "sweep/composite_mean": float(comps.mean()),
                 "sweep/composite_std": float(comps.std()),
                 **{f"sweep/{k}": v for k, v in m.items()}})
        wandb.finish()

        return composite

    except Exception as e:
        wandb.finish(exit_code=1)
        print(f"Trial {trial.number} falló: {e}")
        return 9999.0

    finally:
        # Libera la memoria GPU de este trial para que NO se acumule entre trials (la causa
        # del crash del kernel): del de las referencias + clear_caches (ejecutables compilados
        # del jit(vmap)) + gc para forzar la liberación de los buffers de device.
        out = agent_state = train_fn = env = agent_conf = mh = traj = custom_conf = factory = None
        jax.clear_caches()
        gc.collect()


In [5]:
# ==========================================
# FASE 2: EJECUTAR OPTIMIZACIÓN
# ==========================================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

print("Iniciando la Fase 2: Búsqueda de parámetros de recompensa (métrica debug_agent)...")

# TPE con semilla (reproducible) + arranque aleatorio para sembrar bien el espacio (9 dims).
sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=10)

# Persistencia SQLite. ESTUDIO NUEVO (_v3 / db nueva) a propósito: la "Metric for Sweep" cambió
# de escala (baseline 3.53 -> 6.03, re-baseline a 13-24-37) y el espacio gana una dimensión
# (qvel_exp), así que NO se puede reanudar `ppo_phase2_rewards_v2` / optuna_phase2_v2.db ->
# mezclaría objetivos incomparables y corrompería el TPE.
study = optuna.create_study(
    direction="minimize",
    study_name="ppo_phase2_rewards_v3",
    storage=f"sqlite:///{paths.OPTUNA_DIR / 'optuna_phase2_v3.db'}",
    load_if_exists=True,
    sampler=sampler,
)

# --- WARM-START: sembrar el TPE con las mejores recetas conocidas (solo en el estudio vacío;
# el guard evita re-encolar duplicados si esta celda se re-ejecuta tras una caída). Los nombres
# son los del espacio de búsqueda (qpos_w, no qpos_w_sum). Garantiza que el mejor trial del
# estudio sea >= la config actual, y le da al TPE un punto de partida en la región buena.
#   1. conf.yaml actual = receta anti-shuffle del run 17-19-49 (speed_ratio 0.985, el agente
#      que SÍ camina). rquat_w/rvel_w están exactamente en el techo de su rango (0.3/0.15) ->
#      encolable, pero el TPE no podrá explorar por encima.
#   2-3. Top-2 de la Fase 2 v2 (trials #30/#27): tracking articular excelente pero medidos por
#      la métrica ciega a velocidad y SIN el término rootvel (no existía). Con rootvel ahora
#      fijo en 0.5/4.0 vale la pena re-evaluarlas: pueden trackear bien Y avanzar.
#      qvel_exp no existía en v2 -> se rellena con el 0.25 de conf.yaml.
if len(study.trials) == 0:
    study.enqueue_trial(
        {"qpos_w": 0.27, "rpos_w": 0.28, "qpos_exp": 8.46, "rquat_w": 0.30, "qvel_w": 0.38,
         "rvel_w": 0.15, "qvel_exp": 0.25, "act_rate": 0.002, "torque": 0.008},
        user_attrs={"warm_start": "conf.yaml 2026-06-11 (run 17-19-49, anti-shuffle)"})
    study.enqueue_trial(
        {"qpos_w": 0.6476577100207547, "rpos_w": 0.7934846865576968, "qpos_exp": 8.037282492646625,
         "rquat_w": 0.06862811139825759, "qvel_w": 0.11245553418335238, "rvel_w": 0.04361353766346731,
         "qvel_exp": 0.25, "act_rate": 0.003148258802660846, "torque": 0.0002924499171072347},
        user_attrs={"warm_start": "v2 trial #30 (best v2; qvel_exp filled with 0.25)"})
    study.enqueue_trial(
        {"qpos_w": 0.7100388026647615, "rpos_w": 0.7009224331478044, "qpos_exp": 7.468261365203751,
         "rquat_w": 0.07033682297275097, "qvel_w": 0.18696023827805258, "rvel_w": 0.05559411633491913,
         "qvel_exp": 0.25, "act_rate": 0.008339676072888462, "torque": 0.0005955628060091181},
        user_attrs={"warm_start": "v2 trial #27 (2nd best v2; qvel_exp filled with 0.25)"})
    print("Warm-start: 3 trials encolados (conf.yaml + v2 #30 + v2 #27).")
else:
    print(f"Estudio existente con {len(study.trials)} trials -> no se encola warm-start.")

# 9 hiperparámetros de recompensa -> 30 trials (los 3 primeros = warm-start de arriba).
# Proxy de 20M pasos + rollout de diagnóstico/trial.
study.optimize(objective, n_trials=30)

print("\n¡Búsqueda de Recompensas Finalizada!")
print(f"Mejor métrica compuesta: {study.best_value:.4f}   (baseline 13-24-37 = 6.03; menor es mejor)")
print("Mejores parámetros de recompensa encontrados:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")
# Sub-métricas del mejor trial (qué mejoró: peores articulaciones, supervivencia, suavidad...)
print("Sub-métricas del mejor trial:")
for k, v in study.best_trial.user_attrs.items():
    print(f"  - {k}: {v}")


Iniciando la Fase 2: Búsqueda de parámetros de recompensa (métrica debug_agent)...


[I 2026-06-11 20:38:20,264] A new study created in RDB with name: ppo_phase2_rewards_v3
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ndam/.netrc.


Warm-start: 3 trials encolados (conf.yaml + v2 #30 + v2 #27).


/home/ndam/Desktop/workspace/loco-mujoco-seneca/loco_mujoco/core/mujoco_mjx.py:82: UserWarning: mjwarp is experimental and may not work with all environments. Resetting is error prone. Training might work, but will not be optimal.
  warnings.warn(


Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 5070" (11 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/ndam/.cache/warp/1.12.0
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 0.29 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 95a0b8b load on device 'cuda:0' took 2.99 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 0.30 ms  (cached)
Module _nxn_broadphase__locals__kernel_c763f4cb c763f4c load on device 'cuda:0' took 0.29 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_d126b47c d126b47 load on device 'cuda:0' took 0.72 ms  (cached)
Mod

...

/home/ndam/Desktop/workspace/loco-mujoco-seneca/loco_mujoco/core/mujoco_mjx.py:82: UserWarning: mjwarp is experimental and may not work with all environments. Resetting is error prone. Training might work, but will not be optimal.
  warnings.warn(


[I 2026-06-11 23:53:55,491] Trial 28 finished with value: 10.196636056190963 and parameters: {'qpos_w': 0.7896454232991915, 'rpos_w': 0.5896910549670327, 'qpos_exp': 3.032704909286421, 'rquat_w': 0.18002744291231687, 'qvel_w': 0.2196071752680579, 'rvel_w': 0.11456075472938791, 'qvel_exp': 0.15583934879538802, 'act_rate': 0.015933450743003123, 'torque': 0.00014037793808323592}. Best is trial 17 with value: 8.523125301928165.


/home/ndam/Desktop/workspace/loco-mujoco-seneca/loco_mujoco/core/mujoco_mjx.py:82: UserWarning: mjwarp is experimental and may not work with all environments. Resetting is error prone. Training might work, but will not be optimal.
  warnings.warn(


[I 2026-06-11 23:58:59,366] Trial 29 finished with value: 8.794846800829793 and parameters: {'qpos_w': 0.7388160309902576, 'rpos_w': 0.700808189159456, 'qpos_exp': 5.841795118618355, 'rquat_w': 0.2362811423432664, 'qvel_w': 0.2769336356474311, 'rvel_w': 0.1485825466912636, 'qvel_exp': 0.7717363629540582, 'act_rate': 0.0015308171332932335, 'torque': 0.0057131408373543155}. Best is trial 17 with value: 8.523125301928165.



¡Búsqueda de Recompensas Finalizada!
Mejor métrica compuesta: 8.5231   (baseline 13-24-37 = 6.03; menor es mejor)
Mejores parámetros de recompensa encontrados:
  - qpos_w: 0.4792990831842575
  - rpos_w: 0.6360403640349603
  - qpos_exp: 5.9682311184743835
  - rquat_w: 0.2720356175290966
  - qvel_w: 0.34248070562463884
  - rvel_w: 0.13087520439349054
  - qvel_exp: 1.0194111087248845
  - act_rate: 0.0025146232695822477
  - torque: 0.009895445299195107
Sub-métricas del mejor trial:
  - sweep/act_rate: 0.02288103085583772
  - sweep/bad_death_frac: 0.0
  - sweep/composite_std: 0.0
  - sweep/eplen_frac: 0.97265625
  - sweep/height_rms: 0.10082989117622689
  - sweep/joint_rms_max: 21.883030147908347
  - sweep/joint_rms_mean: 12.70673671463594
  - sweep/joint_rms_worst: 18.60422743350841
  - sweep/speed_ratio: 0.26405495405197144
  - sweep/surv: 0.96875


In [6]:
# ==========================================
# CELDA 5: GRÁFICOS INTERACTIVOS E IMPRESIÓN
# ==========================================
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate

fig1 = plot_optimization_history(study)
fig1.show()

fig2 = plot_param_importances(study)
fig2.show()

fig3 = plot_parallel_coordinate(study)
fig3.show()

print("\nMejores parámetros de recompensa encontrados:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")

# --- TABLA: TOP 5 TRIALS DE LA FASE 2 (menor composite = mejor) ---
import pandas as pd

# Solo trials COMPLETOS y sin el centinela de fallo (el objective devuelve 9999.0 si el trial falla).
validos = [t for t in study.trials
           if t.state == optuna.trial.TrialState.COMPLETE
           and t.value is not None and t.value < 9999.0]
top5 = sorted(validos, key=lambda t: t.value)[:5]

top5_df = pd.DataFrame(
    [{"trial": t.number, "composite": t.value, **t.params} for t in top5]
).set_index("trial")

print(f"\nTop 5 trials de la Fase 2 ('{study.study_name}', {len(validos)} trials válidos):")
# Precisión completa: repr(float) imprime todos los dígitos almacenados (copiables a conf.yaml).
with pd.option_context("display.float_format", lambda v: repr(v)):
    display(top5_df)


Mejores parámetros de recompensa encontrados:
  - qpos_w: 0.4792990831842575
  - rpos_w: 0.6360403640349603
  - qpos_exp: 5.9682311184743835
  - rquat_w: 0.2720356175290966
  - qvel_w: 0.34248070562463884
  - rvel_w: 0.13087520439349054
  - qvel_exp: 1.0194111087248845
  - act_rate: 0.0025146232695822477
  - torque: 0.009895445299195107

Top 5 trials de la Fase 2 ('ppo_phase2_rewards_v3', 30 trials válidos):


,composite,qpos_w,rpos_w,qpos_exp,rquat_w,qvel_w,rvel_w,qvel_exp,act_rate,torque
trial,,,,,,,,,,
17,np.float64(8.523125301928165),np.float64(0.4792990831842575),np.float64(0.6360403640349603),np.float64(5.9682311184743835),np.float64(0.2720356175290966),np.float64(0.34248070562463884),np.float64(0.13087520439349054),np.float64(1.0194111087248845),np.float64(0.0025146232695822477),np.float64(0.009895445299195107)
25,np.float64(8.564043568109167),np.float64(0.735448265373559),np.float64(0.5159927777939307),np.float64(4.478594790324941),np.float64(0.237257358506749),np.float64(0.26311861613808907),np.float64(0.11088768928104084),np.float64(0.7552796797201712),np.float64(0.001356370061715294),np.float64(0.005259020803143462)
11,np.float64(8.587917135850605),np.float64(0.7947962873603129),np.float64(0.3286217570819732),np.float64(5.51959363622626),np.float64(0.23286744473382512),np.float64(0.2814655091808011),np.float64(0.1495910681765063),np.float64(0.5450853190672575),np.float64(0.003526104284733903),np.float64(0.009902608809902094)
21,np.float64(8.658436649240619),np.float64(0.23651656069427546),np.float64(0.40633033344208774),np.float64(9.016192824460568),np.float64(0.2848686885601893),np.float64(0.3907001993599723),np.float64(0.13625933744235827),np.float64(0.3264302129536235),np.float64(0.0021045958620912763),np.float64(0.009701324403298819)
0,np.float64(8.67761316665044),np.float64(0.27),np.float64(0.28),np.float64(8.46),np.float64(0.3),np.float64(0.38),np.float64(0.15),np.float64(0.25),np.float64(0.002),np.float64(0.008)
